# ML in Action: Regression, Clustering & Classification
### AI for Climate & NatureTech — [Dr. Omer Tzuk](https://omertzuk.github.io/)

**GitHub repository:** [AI_for_Climate_and_Nature_Tech_tutorial](https://github.com/omertzuk/AI_for_Climate_and_Nature_Tech_tutorial)

**Datasets** (available in the repository's `datasets/` folder in CSV and Excel formats):
- 🌲 Israeli Forest Monitoring (200 plots) → Regression & Clustering
- 🌊 Water Body Ecological Status (800 rivers) → Classification

**To run:** upload both CSV files to this Colab session, or load directly from GitHub:
```python
BASE = "https://raw.githubusercontent.com/omertzuk/AI_for_Climate_and_Nature_Tech_tutorial/main/datasets/"
df_forest = pd.read_csv(BASE + "israeli_forest_monitoring_dataset.csv")
df_water = pd.read_csv(BASE + "water_body_ecological_status_dataset.csv")
```

# Part 1 — Regression: Predicting Carbon Stock

## 1.1 Load the forest dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.05)

In [ ]:
df_forest = pd.read_csv("israeli_forest_monitoring_dataset.csv")
df_forest.head()

In [ ]:
df_forest = pd.read_excel("israeli_forest_monitoring_dataset.xlsx")
df_forest.head()

## 1.2 Train a Random Forest to predict carbon stock

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

FEATURES = [
    'annual_rainfall_mm', 'mean_temp_c', 'soil_moisture_index',
    'ndvi', 'canopy_cover_pct', 'elevation_m', 'slope_deg',
    'stand_age_years', 'dist_settlement_km',
    'fires_in_past_10yr', 'fire_risk_score'
]
TARGET = 'carbon_stock_tons_per_ha'

X = df_forest[FEATURES]
y = df_forest[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

## 1.3 Predicted vs Actual + Feature Importance + Climate Scenario

In [ ]:

X_warm = X_test.copy()
X_warm['mean_temp_c'] = X_warm['mean_temp_c'] + 2.0
y_pred_warm = model.predict(X_warm)
delta = y_pred_warm.mean() - y_pred.mean()

importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig = plt.figure(figsize=(10, 18))
gs = gridspec.GridSpec(3, 1, figure=fig, hspace=0.35)

ax1 = fig.add_subplot(gs[0])
vmin, vmax = min(y_test.min(), y_pred.min()) - 2, max(y_test.max(), y_pred.max()) + 2
ax1.scatter(y_test, y_pred, alpha=0.65, color='#2E7D32', edgecolors='white', linewidths=0.4, s=55)
ax1.plot([vmin, vmax], [vmin, vmax], '--', color='#BF360C', linewidth=1.8, label='Perfect prediction')
ax1.set_xlabel("Measured carbon stock (t/ha)")
ax1.set_ylabel("Model prediction (t/ha)")
ax1.set_title(f"Predicted vs Actual — R² = {r2:.2f}", fontsize=12, fontweight='bold')
ax1.legend()
ax1.set_xlim(vmin, vmax); ax1.set_ylim(vmin, vmax)
ax1.grid(True, alpha=0.3)

ax2 = fig.add_subplot(gs[1])
colors = ['#1B5E20' if imp >= importances.quantile(0.75) else '#81C784' for imp in importances.values]
ax2.barh(importances.index, importances.values, color=colors, edgecolor='white')
ax2.set_xlabel("Importance score")
ax2.set_title("Feature Importance — What drives carbon storage?", fontsize=12, fontweight='bold')
ax2.grid(True, axis='x', alpha=0.3)

ax3 = fig.add_subplot(gs[2])
ax3.scatter(y_pred, y_pred_warm, alpha=0.65, color='#E65100', edgecolors='white', linewidths=0.4, s=55)
ax3.plot([y_pred.min(), y_pred.max()], [y_pred.min(), y_pred.max()], '--', color='#607D8B', linewidth=1.8, label='No change')
ax3.set_xlabel("Predicted carbon — current climate (t/ha)")
ax3.set_ylabel("Predicted carbon — +2°C scenario (t/ha)")
ax3.set_title(f"Climate Scenario: +2°C warming → {delta:+.1f} t/ha average change", fontsize=12, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

fig.subplots_adjust(left=0.12, right=0.95, top=0.96, bottom=0.05, hspace=0.40)
plt.show()

# Part 2 — Clustering: Discovering Ecological Regions

## 2.1 K-Means: Elbow Method

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
ENV_FEATURES = [
    'annual_rainfall_mm', 'mean_temp_c', 'soil_moisture_index',
    'ndvi', 'canopy_cover_pct', 'elevation_m', 'slope_deg',
    'stand_age_years', 'dist_settlement_km', 'fires_in_past_10yr',
    'fire_risk_score'
]

In [ ]:
X_clust = df_forest[ENV_FEATURES].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clust)

K_RANGE = range(1, 11)
inertias = [KMeans(n_clusters=k, random_state=42, n_init=20).fit(X_scaled).inertia_ for k in K_RANGE]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(K_RANGE), inertias, marker='o', markersize=8, linewidth=2.5,
        color='#2E7D32', markerfacecolor='white', markeredgewidth=2)
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Inertia")
ax.set_title("Elbow Method — How many ecological groups?", fontsize=12, fontweight='bold')
ax.set_xticks(list(K_RANGE))
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.show()

## 2.2 K-Means (k=5): Clusters vs True Regions

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

kmeans = KMeans(n_clusters=5, random_state=42, n_init=20)
cluster_labels = kmeans.fit_predict(X_scaled)

In [ ]:
CLUSTER_COLORS = ['#1976D2', '#388E3C', '#F57C00', '#7B1FA2', '#D32F2F']
REGION_COLORS = {
    'Upper Galilee': '#1976D2', 'Carmel': '#388E3C',
    'Judean Hills': '#F57C00', 'Lower Galilee': '#7B1FA2', 'Yatir Edge': '#D32F2F'
}

df_forest['cluster'] = cluster_labels
cluster_profile = df_forest.groupby('cluster')[
    ['annual_rainfall_mm', 'mean_temp_c', 'fire_risk_score', 'ndvi', 'canopy_cover_pct', 'elevation_m']
].mean().round(1)
cluster_region = df_forest.groupby('cluster')['region'].agg(lambda x: x.value_counts().index[0])
cluster_profile['dominant_region'] = cluster_region

In [ ]:
fig = plt.figure(figsize=(18, 6))
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.38)

ax1 = fig.add_subplot(gs[0])
for c in range(5):
    mask = cluster_labels == c
    ax1.scatter(X_pca[mask, 0], X_pca[mask, 1], color=CLUSTER_COLORS[c], alpha=0.75, s=50,
                edgecolors='white', linewidths=0.4, label=f"Cluster {c}")
centers_pca = pca.transform(kmeans.cluster_centers_)
ax1.scatter(centers_pca[:, 0], centers_pca[:, 1], marker='*', s=280, color='black', label='Center')
ax1.set_xlabel("PCA axis 1"); ax1.set_ylabel("PCA axis 2")
ax1.set_title("K-Means clusters (no labels used)", fontsize=11, fontweight='bold')
ax1.legend(fontsize=7)
ax1.grid(True, alpha=0.25)

ax2 = fig.add_subplot(gs[1])
for region, color in REGION_COLORS.items():
    mask = df_forest['region'] == region
    ax2.scatter(X_pca[mask, 0], X_pca[mask, 1], color=color, alpha=0.75, s=50,
                edgecolors='white', linewidths=0.4, label=region)
ax2.set_xlabel("PCA axis 1"); ax2.set_ylabel("PCA axis 2")
ax2.set_title("The reveal — true ecological regions", fontsize=11, fontweight='bold')
ax2.legend(fontsize=7)
ax2.grid(True, alpha=0.25)

ax3 = fig.add_subplot(gs[2])
profile_plot = cluster_profile[['annual_rainfall_mm', 'mean_temp_c', 'fire_risk_score', 'ndvi', 'canopy_cover_pct', 'elevation_m']].copy()
profile_norm = (profile_plot - profile_plot.min()) / (profile_plot.max() - profile_plot.min())
profile_norm.index = [f"Cluster {i}\n({cluster_region[i].split()[0]})" for i in profile_norm.index]
profile_norm.columns = ['Rainfall', 'Temp', 'Fire risk', 'NDVI', 'Canopy', 'Elevation']
sns.heatmap(profile_norm, annot=profile_plot.values, fmt='.0f', cmap='RdYlGn_r', ax=ax3,
            linewidths=0.5, annot_kws={'size': 8})
ax3.set_title("Ecological fingerprint per cluster", fontsize=11, fontweight='bold')
ax3.set_xticklabels(ax3.get_xticklabels(), rotation=35, ha='right', fontsize=9)

fig.subplots_adjust(left=0.04, right=0.98, top=0.88, bottom=0.08, wspace=0.38)
plt.show()

## 2.3 Hierarchical Clustering — Dendrogram

In [ ]:
import matplotlib.patches as mpatches
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from collections import Counter

In [ ]:
np.random.seed(42)
N_SAMPLE = 60
idx_sample = (
    df_forest.groupby('region', group_keys=False)
    .apply(lambda g: g.sample(min(len(g), N_SAMPLE // 5), random_state=42))
    .index
)
X_sub = X_scaled[df_forest.index.get_indexer(idx_sample)]
labels_sub = df_forest.loc[idx_sample, 'region'].values

Z = linkage(X_sub, method='ward', metric='euclidean')
k_cut = 5
flat_clusters = fcluster(Z, k_cut, criterion='maxclust')

SHORT_LABELS = {'Upper Galilee': 'UG', 'Carmel': 'Ca', 'Judean Hills': 'JH',
                'Lower Galilee': 'LG', 'Yatir Edge': 'Ya'}
leaf_labels = [SHORT_LABELS[r] for r in labels_sub]

def make_link_color_func(Z, labels_sub, region_colors):
    n_leaves = len(labels_sub)
    node_leaves = {i: {i} for i in range(n_leaves)}
    for step, (left, right, _, _) in enumerate(Z):
        node_leaves[n_leaves + step] = node_leaves[int(left)] | node_leaves[int(right)]
    def link_color_func(node_id):
        leaves = node_leaves.get(node_id, set())
        region_counts = Counter(labels_sub[i] for i in leaves)
        dominant, count = region_counts.most_common(1)[0]
        if count / sum(region_counts.values()) >= 0.60:
            return region_colors[dominant]
        return '#AAAAAA'
    return link_color_func

link_color_func = make_link_color_func(Z, labels_sub, REGION_COLORS)

fig, axes = plt.subplots(1, 2, figsize=(18, 8), gridspec_kw={'width_ratios': [3, 1]})

ax_dendro = axes[0]
dend = dendrogram(Z, ax=ax_dendro, labels=leaf_labels, leaf_rotation=90, leaf_font_size=7,
                  color_threshold=0, link_color_func=link_color_func, above_threshold_color='#AAAAAA')
for lbl_obj, region in zip(ax_dendro.get_xticklabels(), labels_sub):
    lbl_obj.set_color(REGION_COLORS[region])
    lbl_obj.set_fontweight('bold')

cut_height = (Z[-(k_cut - 1), 2] + Z[-(k_cut), 2]) / 2
ax_dendro.axhline(y=cut_height, color='#BF360C', linewidth=2, linestyle='--')
ax_dendro.text(len(labels_sub) * 0.72, cut_height * 1.04, f'Cut → {k_cut} clusters',
               color='#BF360C', fontsize=10, fontweight='bold')
ax_dendro.set_title("Hierarchical Clustering Dendrogram", fontsize=12, fontweight='bold')
ax_dendro.set_xlabel("Forest plots (UG=Upper Galilee, Ca=Carmel, JH=Judean Hills, LG=Lower Galilee, Ya=Yatir)")
ax_dendro.set_ylabel("Merge distance (Ward linkage)")
ax_dendro.legend(handles=[mpatches.Patch(color=c, label=r) for r, c in REGION_COLORS.items()],
                 title='Region', fontsize=8, title_fontsize=8, loc='upper right')

ax_bar = axes[1]
bar_data = {}
for c in sorted(set(flat_clusters)):
    bar_data[f'Cluster {c}'] = Counter(labels_sub[flat_clusters == c])
bar_bottoms = np.zeros(len(bar_data))
for region in REGION_COLORS:
    counts = [bar_data[c].get(region, 0) for c in bar_data]
    ax_bar.barh(list(bar_data.keys()), counts, left=bar_bottoms,
                color=REGION_COLORS[region], label=region, edgecolor='white', linewidth=0.8)
    bar_bottoms += np.array(counts)
ax_bar.set_xlabel("Number of plots")
ax_bar.set_title("Cluster\ncomposition", fontsize=11, fontweight='bold')
ax_bar.invert_yaxis()
ax_bar.legend(title='Region', fontsize=7, title_fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')

fig.subplots_adjust(left=0.06, right=0.88, top=0.92, bottom=0.10, wspace=0.15)
plt.show()

# Part 3 — Classification: Predicting Ecological Status of Rivers

## Dataset description

---
Synthetic Water Body Ecological Status Dataset
Based on: Martyszunis, Loga & Przeździecki (2024), Scientific Reports
"Using ML for assessment of ecological status of unmonitored waters in Poland" - [link](https://doi.org/10.1038/s41598-024-74511-4)

---

AI for Climate & NatureTech — Dr. Omer Tzuk
Colab Demo: Classification of ecological status from catchment pressures

This script generates a realistic synthetic dataset of ~800 river water bodies
modeled on the WFD (Water Framework Directive) framework used across Europe.

Key design choices matching the paper:
  - 5 ecological status classes: high, good, moderate, poor, bad
  - Severe class imbalance: only ~7% "at least good", ~93% "below good"
  - "moderate" class is hardest to separate (paper's key finding)
  - Features are anthropogenic PRESSURES, not water quality measurements
    (because the goal is to predict status for UNMONITORED water bodies)
  - 176 features in the paper → simplified to 18 interpretable features
  - Split into "monitored" (for training) and "unmonitored" (for prediction)

The dataset mimics Polish river systems but the concepts are universal
for any WFD country or freshwater monitoring context.


| Column | Description |
|--------|-------------|
| `wb_id` | Unique identifier for each water body |
| `river_basin` | Major river basin the water body belongs to (Vistula, Oder, Coastal, Dniester) |
| `is_monitored` | Whether this water body has field monitoring data (True) or needs ML prediction (False) |
| **Hydromorphological** | |
| `basin_area_km2` | Total catchment drainage area in km² — larger basins accumulate more pressures |
| `stream_length_km` | Length of the river segment in km |
| `mean_annual_flow_m3s` | Average river discharge in m³/s — higher flow dilutes pollutants |
| **Land use pressures** | |
| `pct_agricultural_land` | Percentage of catchment under agriculture — source of nutrient runoff |
| `pct_urban_land` | Percentage of catchment that is built-up — source of stormwater and sewage |
| `pct_forest_land` | Percentage of catchment covered by forest — protective, filters runoff |
| `population_density_per_km2` | People per km² in the catchment — proxy for total human pressure |
| `impervious_surface_pct` | Percentage of sealed/paved surfaces — drives stormwater runoff intensity |
| **Wastewater pressures** | |
| `wwtp_discharge_m3_day` | Volume of treated wastewater entering the river in m³/day |
| `wwtp_treatment_level` | Quality of wastewater treatment: 1 = primary (minimal), 2 = secondary, 3 = tertiary (best) |
| `num_industrial_sources` | Count of industrial discharge points in the catchment |
| `industrial_bod5_kg_day` | Biological oxygen demand from industrial discharges in kg/day — measures organic pollution load |
| `stormwater_overflow_count` | Number of combined sewer overflow events — untreated sewage entering the river during storms |
| **Agricultural pressures** | |
| `agricultural_runoff_tn_kg` | Total nitrogen load from agricultural runoff in kg — drives eutrophication |
| `agricultural_runoff_tp_kg` | Total phosphorus load from agricultural runoff in kg — drives algal blooms |
| `livestock_density_lu_km2` | Livestock units per km² — source of manure and nutrient pollution |
| **Protective factors** | |
| `riparian_buffer_pct` | Percentage of riverbank with natural vegetation — filters pollutants before they reach the water |
| `dam_count_upstream` | Number of dams upstream — alters flow regime and sediment transport |
| **Derived features** | |
| `pressure_intensity` | Wastewater discharge normalized by river flow — measures how overwhelmed the river is |
| `nutrient_load_per_km2` | Total nutrient runoff (N+P) per km² of catchment — normalized pollution density |
| **Noise features** | |
| `sampling_year` | Year of the monitoring cycle (2016–2021) — should NOT predict status |
| `water_body_type_code` | Administrative river type classification (1–21) — weak predictor at best |
| `elevation_m` | Mean catchment elevation in meters |
| `catchment_perimeter_km` | Perimeter of the catchment boundary in km |
| `avg_channel_width_m` | Average river channel width in meters |
| **Target variables** | |
| `ecological_status` | WFD ecological status: high, good, moderate, poor, or bad (5 classes) |
| `status_binary` | Simplified binary target: "at_least_good" vs "below_good" (Paper's Variant 1) |
| `status_numeric` | Ordinal encoding: 5 = high, 4 = good, 3 = moderate, 2 = poor, 1 = bad |

## Loading the dataset

In [ ]:
df_water = pd.read_csv("water_body_ecological_status_dataset.csv")

In [ ]:
df_water = pd.read_excel("water_body_ecological_status_dataset.xlsx")

In [ ]:
df_mon = df_water[df_water["is_monitored"] == True].copy()
df_unmon = df_water[df_water["is_monitored"] == False].copy()

In [ ]:
feature_cols = [
    "basin_area_km2", "stream_length_km", "mean_annual_flow_m3s",
    "pct_agricultural_land", "pct_urban_land", "pct_forest_land",
    "population_density_per_km2", "impervious_surface_pct",
    "wwtp_discharge_m3_day", "wwtp_treatment_level",
    "num_industrial_sources", "industrial_bod5_kg_day",
    "stormwater_overflow_count",
    "agricultural_runoff_tn_kg", "agricultural_runoff_tp_kg",
    "livestock_density_lu_km2",
    "riparian_buffer_pct", "dam_count_upstream",
    "pressure_intensity", "nutrient_load_per_km2",
]

df_mon["target_binary"] = df_mon["ecological_status"].map(
    {"high": 1, "good": 1, "moderate": 0, "poor": 0, "bad": 0}
)

X_w = df_mon[feature_cols].values
y_w = df_mon["target_binary"].values

X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_w, y_w, test_size=0.25, random_state=42, stratify=y_w
)

In [ ]:
scaler_w = StandardScaler()
X_train_ws = scaler_w.fit_transform(X_train_w)
X_test_ws = scaler_w.transform(X_test_w)

df_water.head(20)

## 3.2 Random Forest — Binary Classification + Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf_binary = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
rf_binary.fit(X_train_ws, y_train_w)
y_pred_w = rf_binary.predict(X_test_ws)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test_w, y_pred_w, labels=[1, 0])
sns.heatmap(cm, annot=True, fmt="d", cmap="YlOrRd", ax=axes[0],
            xticklabels=["Good+", "Below Good"], yticklabels=["Good+", "Below Good"],
            annot_kws={"size": 18})
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_title(f"Confusion Matrix (counts) — Accuracy: {accuracy_score(y_test_w, y_pred_w)*100:.0f}%",
                  fontsize=11, fontweight='bold')

cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
sns.heatmap(cm_pct, annot=True, fmt=".0f", cmap="YlOrRd", ax=axes[1],
            xticklabels=["Good+", "Below Good"], yticklabels=["Good+", "Below Good"],
            annot_kws={"size": 18}, vmin=0, vmax=100)
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Actual")
axes[1].set_title("Recall per class (%)", fontsize=11, fontweight='bold')

plt.suptitle("Binary Classification: Good+ vs Below-Good", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 3.3 Feature Importance

In [ ]:
importances_w = rf_binary.feature_importances_
feat_imp_w = pd.Series(importances_w, index=feature_cols).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
colors_w = ["#2ecc71" if f in ("riparian_buffer_pct", "pct_forest_land", "wwtp_treatment_level")
            else "#e74c3c" if f in ("pct_urban_land", "wwtp_discharge_m3_day", "num_industrial_sources")
            else "#3498db" for f in feat_imp_w.index]
feat_imp_w.plot(kind="barh", ax=ax, color=colors_w, edgecolor="white", linewidth=0.5)
ax.set_xlabel("Importance Score")
ax.set_title("Feature Importance — Green = protective | Red = pressure", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 3.4 Compare Algorithms

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

In [ ]:
algorithms = {
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "KNN (k=5)":     KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes":   GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42),
    "SVM (RBF)":     SVC(kernel="rbf", class_weight="balanced", random_state=42),
}

algo_results = {}
for name, mdl in algorithms.items():
    mdl.fit(X_train_ws, y_train_w)
    yp = mdl.predict(X_test_ws)
    algo_results[name] = {
        "accuracy": accuracy_score(y_test_w, yp),
        "recall_good": (yp[y_test_w == 1] == 1).mean() if (y_test_w == 1).sum() > 0 else 0,
        "recall_below": (yp[y_test_w == 0] == 0).mean() if (y_test_w == 0).sum() > 0 else 0,
    }

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
algo_names = list(algo_results.keys())
x = np.arange(len(algo_names))
width = 0.25
bars1 = ax.bar(x - width, [algo_results[a]["accuracy"]*100 for a in algo_names], width, label="Accuracy", color="#3498db", edgecolor="white")
bars2 = ax.bar(x, [algo_results[a]["recall_good"]*100 for a in algo_names], width, label="Recall: Good+", color="#2ecc71", edgecolor="white")
bars3 = ax.bar(x + width, [algo_results[a]["recall_below"]*100 for a in algo_names], width, label="Recall: Below Good", color="#e74c3c", edgecolor="white")
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{bar.get_height():.0f}%", ha="center", va="bottom", fontsize=8, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(algo_names, fontsize=10)
ax.set_ylabel("Score (%)"); ax.set_ylim(0, 110)
ax.set_title("Algorithm Comparison — Binary Classification", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()